# Train a model — SeqReach1D

Runs top to bottom. `src(x)` prints the source of `x`; the comments in the source are the docs.

In [1]:
import inspect, json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "seqreach").is_dir())  # .../1D
sys.path.insert(0, str(ROOT))

from seqreach import config as config_mod, env as env_mod, metrics, plots, probes, video
from seqreach.config import Config
from seqreach.env import BatchedSeqReach1D
from seqreach.networks import Agent
from seqreach.ppo import train, load_agent
from seqreach.tracking import read_train_log

src = lambda obj: print(inspect.getsource(obj))
OUT = ROOT / "runs" / "nb_out"; OUT.mkdir(parents=True, exist_ok=True)  # runs/ is gitignored
ROOT

PosixPath('/Users/aghavamp/Desktop/Projects/SeqHorizon_basic/1D')

## 1 · Config — `seqreach/config.py`

In [2]:
cfg = Config()
list(cfg.to_dict())          # sections

['exp_name',
 'seed',
 'torch_threads',
 'plant',
 'task',
 'reward',
 'agent',
 'ppo',
 'video',
 'wandb']

In [11]:
cfg.plant, cfg.task.max_targets, cfg.ppo.gamma, cfg.reward.r_hit

(PlantConfig(dt=0.02, mass=1.0, damping=0.2, u_max=8.0, tau_act=0.05, sigma_sdn=0.05, sigma_add=0.0, sensory_delay=0, x_limit=1.5),
 3,
 0.995,
 10.0)

In [12]:
src(config_mod.PlantConfig)
src(config_mod.TaskConfig)

@dataclass
class PlantConfig:
    """Point-mass plant.  Second order + a first-order activation lag."""

    dt: float = 0.02
    mass: float = 1.0
    damping: float = 0.2
    u_max: float = 8.0
    # First-order activation lag: force cannot change instantaneously (muscle-like).
    # tau_act = 0 collapses to a pure double integrator.  Ablation axis C5.
    tau_act: float = 0.05
    # Signal-dependent motor noise (Harris & Wolpert): sd proportional to |activation|.
    sigma_sdn: float = 0.05
    # Constant additive motor noise.
    sigma_add: float = 0.0
    # Sensory feedback delay in steps.  >0 forces the policy to predict its own state.
    sensory_delay: int = 0
    # Workspace half-width.  The boundary is *inelastic*: contact zeroes velocity and is
    # charged for (reward.w_wall).  An earlier version clipped position and left velocity
    # untouched and unpenalised, which made "drive into the wall and sit" a free resting
    # place and trapped most of the ablation grid there

In [13]:
src(config_mod.RewardConfig)   # the entire task specification the agent ever sees

@dataclass
class RewardConfig:
    """The entire task specification the agent ever sees.

    Note what is absent: no trajectory target, no velocity profile, no via-point, no
    reaction-time term, no preparatory-state term.
    """

    r_hit: float = 10.0
    r_final: float = 20.0
    r_false_start: float = -5.0

    w_effort: float = 0.5  # on (u/u_max)^2 * dt        -- ablation axis C2
    w_rate: float = 0.05  # on ((u - u_prev)/u_max)^2
    w_time: float = 1.0  # per second while targets remain -- ablation axis C3
    w_hold: float = 20.0  # on (x - x0)^2 before the go cue
    w_hold_v: float = 2.0  # on v^2 before the go cue

    # Per step of contact with the workspace boundary.  Sized against the two costs it
    # competes with: time is 0.02/step and shaping pays at most (1-gamma)*coef*|x-T| =
    # 0.024/step out there, so 0.1 makes boundary contact decisively negative rather than
    # break-even.  A competent policy never touches the boundary (measured: baseline and
    #

In [14]:
src(config_mod.PPOConfig)
src(config_mod.AgentConfig)

@dataclass
class PPOConfig:
    """CleanRL PPO hyperparameters (ppo_continuous_action / ppo_atari_lstm)."""

    total_timesteps: int = 2_000_000
    num_envs: int = 32
    num_steps: int = 256
    learning_rate: float = 3e-4
    anneal_lr: bool = True
    # gamma sets how far credit can travel.  Too myopic and the agent cannot "see" the
    # next target's cost even with full visual preview.  Headline ablation axis C7.
    gamma: float = 0.995
    gae_lambda: float = 0.95
    num_minibatches: int = 4
    update_epochs: int = 4
    clip_coef: float = 0.2
    clip_vloss: bool = True
    ent_coef: float = 0.003
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    target_kl: float | None = None
    norm_adv: bool = True
    norm_reward: bool = True
    norm_obs: bool = True
    init_logstd: float = -0.5

@dataclass
class AgentConfig:
    recurrent: bool = True  # GRU; False -> MLP (ablation)
    hidden_size: int = 128
    rnn_size: int = 128
    n_hidden_layers: int = 2
    activat

In [15]:
# dotted overrides -> a new Config; unknown keys raise
cfg = Config().override({"ppo.gamma": 0.97, "reward.w_effort": 0.0, "task.dwell_inter": 25})
cfg.exp_name, cfg.seed = "my_run", 1
cfg.run_name, cfg.ppo.gamma, cfg.reward.w_effort

('my_run_s1', 0.97, 0.0)

In [16]:
cfg.save(OUT / "my_run.yaml")
Config.load(OUT / "my_run.yaml").ppo.gamma
# Config() is the source of truth; start from it unless you want that snapshot.

0.97

## 2 · Environment — `seqreach/env.py`

In [18]:
cfg = Config().override({"plant.sigma_sdn": 0.0})   # noise off -> reproducible
env = BatchedSeqReach1D(cfg, num_envs=4, seed=0)
env.obs_dim, env.act_dim, env.K, env.dt

(16, 1, 3, 0.02)

In [22]:
# observation layout: [x, v, a, go] + per target slot [visible, pending, is_current, rel]
B, S = env_mod._BASE_FEATURES, env_mod._SLOT_FEATURES
NAMES = ["x", "v", "a", "go"] + [f"{f}{k}" for k in range(env.K) for f in ("vis", "pend", "cur", "rel")]

obs = env.reset()
pd.DataFrame(obs, columns=NAMES).round(3)

,x,v,a,go,vis0,pend0,cur0,rel0,vis1,pend1,cur1,rel1,vis2,pend2,cur2,rel2
0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.891,1.0,1.0,0.0,0.562,1.0,1.0,0.0,0.386
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,-0.331,0.0,1.0,0.0,0.000,0.0,1.0,0.0,-0.000
2,0.0,0.0,0.0,0.0,1.0,1.0,1.0,-0.571,1.0,1.0,0.0,0.825,0.0,1.0,0.0,-0.000
3,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.684,0.0,1.0,0.0,0.000,0.0,0.0,0.0,-0.000


In [23]:
src(env_mod.BatchedSeqReach1D._visibility)
src(env_mod.BatchedSeqReach1D._obs)

    def _visibility(self) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Return (visible, pending, is_current) boolean masks of shape (N, K)."""
        slots = np.arange(self.K)[None, :]
        in_seq = slots < self.n_targets[:, None]
        pending = in_seq & (slots >= self.cur[:, None])
        visible = pending & (slots <= (self.cur + self.preview - 1)[:, None])
        is_cur = slots == self.cur[:, None]
        return visible, pending, is_cur & in_seq

    def _obs(self) -> np.ndarray:
        x_hat = self._x_hist[self.d_sens]
        v_hat = self._v_hist[self.d_sens]
        visible, pending, is_cur = self._visibility()

        out = np.empty((self.n, self.obs_dim), dtype=np.float32)
        out[:, 0] = x_hat / _X_SCALE
        out[:, 1] = v_hat / _V_SCALE
        out[:, 2] = self.a / self.cfg.plant.u_max
        out[:, 3] = (self.t >= self.go_step).astype(np.float64)
        rel = (self.targets - x_hat[:, None]) * visible / _X_SCALE
        for k in range(self.K):


In [24]:
# hidden trial state, sampled per trial by _sample_trials()
{"targets": env.targets, "n_targets": env.n_targets, "preview": env.preview,
 "go_step": env.go_step}

{'targets': array([[ 0.89085123,  0.56220372,  0.38631337],
        [-0.33130162,  0.30220093, -0.45513587],
        [-0.57111772,  0.82514454, -0.6457562 ],
        [ 0.68417662,  0.44684731, -0.59930473]]),
 'n_targets': array([3, 3, 3, 2]),
 'preview': array([3, 1, 2, 1]),
 'go_step': array([32, 41, 32, 23])}

In [25]:
src(env_mod.BatchedSeqReach1D._sample_trials)

    def _sample_trials(self, idx: np.ndarray, spec: dict | None = None) -> None:
        """(Re)initialise the trials at ``idx``.  ``spec`` overrides sampling (probes)."""
        if idx.size == 0:
            return
        cfg, t, p = self.cfg, self.cfg.task, self.cfg.plant
        m = idx.size
        spec = spec or {}

        self.x[idx] = t.start_x
        self.v[idx] = 0.0
        self.a[idx] = 0.0
        self.u_prev[idx] = 0.0
        self.t[idx] = 0
        self.cur[idx] = 0
        self.dwell[idx] = 0
        self.ep_return[idx] = 0.0
        self.ep_return_task[idx] = 0.0
        self.ep_len[idx] = 0
        self.ep_hits[idx] = 0
        self.ep_wall[idx] = 0
        self.jump_applied[idx] = False
        self.jump_slot[idx] = -1

        # --- number of targets
        if "n_targets" in spec:
            n_tg = np.broadcast_to(np.asarray(spec["n_targets"]), (m,)).astype(np.int64)
        else:
            w = self._n_target_weights
            n_tg = self.rng.choice(len(w)

In [26]:
# a `spec` pins any of it (this is what probes / videos / experiments do)
obs = env.reset({
    "n_targets":   [2, 2, 3, 1],
    "targets":     [[0.5, -0.4, 0.3]] * 4,
    "preview":     [2, 1, 3, 1],      # how many targets are visible at once
    "delay_steps": [25, 25, 25, 25],  # hold before the go cue (steps; dt = 20 ms)
})
pd.DataFrame(obs, columns=NAMES).round(3)

,x,v,a,go,vis0,pend0,cur0,rel0,vis1,pend1,cur1,rel1,vis2,pend2,cur2,rel2
0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,1.0,1.0,0.0,-0.4,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,0.0,1.0,0.0,-0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,1.0,1.0,0.0,-0.4,1.0,1.0,0.0,0.3
3,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.5,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0


In [27]:
# one step: action is force in [-u_max, u_max], shape (num_envs,)
obs, rew, term, trunc, info = env.step(np.zeros(4))
rew, term, trunc, info.keys()

(array([0., 0., 0., 0.], dtype=float32),
 array([False, False, False, False]),
 array([False, False, False, False]),
 dict_keys([]))

## 3 · Reward — every term is added in `env.step`

In [ ]:
src(env_mod.BatchedSeqReach1D.step)

In [ ]:
def run_trial(cfg, targets=(0.5, -0.4), delay=25, kp=40.0, kd=12.0, seed=0):
    """One trial under a hand-written PD controller (no learning). Records everything."""
    env = BatchedSeqReach1D(cfg, num_envs=1, seed=seed)
    tg = np.zeros(cfg.task.max_targets); tg[:len(targets)] = targets
    obs = env.reset({"n_targets": [len(targets)], "targets": [tg],
                     "preview": [len(targets)], "delay_steps": [delay]})
    out = {"go_step": int(env.go_step[0]), "n_targets": len(targets), "targets": tg}
    rec = {k: [] for k in ("t", "x", "v", "u", "r", "cur", "cur_after")}
    for _ in range(cfg.task.max_steps):
        go   = obs[0, 3]                                        # go cue flag
        rel  = obs[0, B + 3::S] * obs[0, B + 2::S]              # rel offset of the current target
        u    = go * np.clip(kp * rel.sum() - kd * (obs[0, 1] * 2.0), -cfg.plant.u_max, cfg.plant.u_max)
        t_before, cur_before = int(env.t[0]), int(env.cur[0])
        obs, r, term, trunc, info = env.step(np.array([u]))
        done = bool(term[0] or trunc[0])
        for k, val in (("t", t_before), ("cur", cur_before), ("x", env.x[0]), ("v", env.v[0]),
                       ("u", u), ("r", float(r[0])),
                       ("cur_after", int(info["episode"]["cur"][0]) if done else int(env.cur[0]))):
            rec[k].append(val)
        if done:
            out["episode"] = {k: v[0] for k, v in info["episode"].items()}
            break
    return out | {k: np.asarray(v) for k, v in rec.items()}

tr = run_trial(Config().override({"plant.sigma_sdn": 0.0, "reward.shaping": "none"}))
tr["episode"]

In [ ]:
def reward_terms(tr, cfg):
    """Re-derive each reward component from the trajectory — the formulas in env.step."""
    r, p, t = cfg.reward, cfg.plant, cfg.task
    u_prev = np.concatenate([[0.0], tr["u"][:-1]])
    pre_go = tr["t"] < tr["go_step"]
    acq    = tr["cur_after"] > tr["cur"]                      # a target was taken this step
    last   = tr["cur"] == tr["n_targets"] - 1
    return {
        "effort": -r.w_effort * (tr["u"] / p.u_max) ** 2 * p.dt,
        "rate":   -r.w_rate * ((tr["u"] - u_prev) / p.u_max) ** 2,
        "time":   -r.w_time * p.dt * ~pre_go,
        "hold":   -r.w_hold * (tr["x"] - t.start_x) ** 2 * pre_go,
        "hold_v": -r.w_hold_v * tr["v"] ** 2 * pre_go,
        "wall":   -r.w_wall * (np.abs(tr["x"]) >= p.x_limit),
        "hit":    acq * np.where(last, r.r_final, r.r_hit),
    }

cfg_sparse = Config().override({"plant.sigma_sdn": 0.0, "reward.shaping": "none"})
terms = reward_terms(tr, cfg_sparse)
print("matches env reward:", np.allclose(sum(terms.values()), tr["r"], atol=1e-6))
{k: round(float(v.sum()), 3) for k, v in terms.items()} | {"TOTAL": round(float(tr["r"].sum()), 3)}

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
time_s = tr["t"] * cfg_sparse.plant.dt
ax[0].plot(time_s, tr["x"], "k")
for j, tg in enumerate(tr["targets"][:tr["n_targets"]]):
    ax[0].axhspan(tg - cfg_sparse.task.tolerance, tg + cfg_sparse.task.tolerance,
                  color=f"C{j}", alpha=.25)
ax[0].axvline(tr["go_step"] * cfg_sparse.plant.dt, color="g", ls="--")   # go cue
ax[0].set_ylabel("x")
ax[1].plot(time_s, tr["u"], "k"); ax[1].set_ylabel("u (force)")
for k, v in terms.items():
    ax[2].plot(time_s, np.cumsum(v), label=k)
ax[2].legend(ncol=4, fontsize=8); ax[2].set_ylabel("cumulative reward"); ax[2].set_xlabel("s")
plt.tight_layout()

In [ ]:
# same behaviour, different reward config -> different return.
# pbrs is policy-invariant shaping (a learning aid); greedy is the naive distance cost.
for name, over in {
    "default (pbrs)":  {},
    "sparse":          {"reward.shaping": "none"},
    "greedy":          {"reward.shaping": "greedy"},
    "no effort cost":  {"reward.w_effort": 0.0, "reward.w_rate": 0.0},
    "no time cost":    {"reward.w_time": 0.0},
    "cheap targets":   {"reward.r_hit": 1.0, "reward.r_final": 2.0},
}.items():
    c = Config().override({"plant.sigma_sdn": 0.0} | over)
    t_ = run_trial(c)
    print(f"{name:>16}  return {t_['episode']['r']:8.2f}   task-only {t_['episode']['r_task']:8.2f}")

In [ ]:
src(env_mod.BatchedSeqReach1D._potential)   # Phi for potential-based shaping

## 4 · Agent — `seqreach/networks.py`

In [ ]:
agent = Agent(env.obs_dim, env.act_dim, Config().agent, init_logstd=-0.5)
agent

In [ ]:
state = agent.initial_state(4, torch.device("cpu"))          # GRU hidden state (1, N, 128)
done  = torch.zeros(4)
action, logprob, entropy, value, state = agent.get_action_and_value(
    torch.tensor(obs), state, done)
action.shape, value.shape, state.shape

In [ ]:
# actions are in [-1, 1]-ish and scaled by u_max in the training loop / probes
mean, _, _, value, _ = agent.get_action_and_value(torch.tensor(obs), state, done, deterministic=True)
(mean.detach().numpy().ravel() * Config().plant.u_max).round(3)

In [ ]:
src(Agent.get_states)   # how the recurrent state is zeroed at episode boundaries

## 5 · Train — `seqreach/ppo.py`

In [ ]:
# src(train)     # the whole PPO loop (CleanRL port); read it in seqreach/ppo.py

In [ ]:
toy = Config().override({
    "ppo.total_timesteps": 40_000,     # ~1 min on CPU.  A real run is 3_000_000.
    "ppo.num_envs": 8,                 # must divide by ppo.num_minibatches
    "ppo.num_steps": 128,              # batch = num_envs * num_steps
    "task.curriculum_steps": 20_000,   # scaled down with the run
    "video.enabled": False,            # True -> a GIF every video.every_iters iterations
    "wandb.enabled": False,            # True for real runs
})
toy.exp_name, toy.seed = "nb_toy", 1

run_dir = ROOT / "runs" / toy.run_name          # runs/nb_toy_s1
agent, obs_rms, run_dir = train(toy, run_dir, log_every=5)
sorted(p.name for p in run_dir.iterdir())

In [ ]:
log = pd.DataFrame(read_train_log(run_dir))     # runs/<name>/train_log.jsonl
log.tail(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
log.plot(x="global_step", y="episodic_return_task", ax=ax[0])   # watch this, not episodic_return
log.plot(x="global_step", y=["success_rate", "hit_fraction"], ax=ax[1])
log.plot(x="global_step", y="explained_variance", ax=ax[2])
plt.tight_layout()

Same thing from the shell (`scripts/train.py`):

```bash
python 1D/scripts/train.py --name baseline --timesteps 3000000 --wandb
```

```bash
python 1D/scripts/train.py --name gamma097_sparse --set ppo.gamma=0.97 reward.shaping=none --wandb
```

## 6 · Load a trained policy and measure it — `seqreach/probes.py`

In [ ]:
agent, obs_rms, cfg = load_agent(ROOT / "runs" / "baseline_s1")   # checkpoint.pt + obs_rms + config
cfg.exp_name, cfg.ppo.gamma, cfg.ppo.total_timesteps

In [ ]:
pcfg = probes.probe_config(cfg)      # motor noise off, jumps off -> deterministic replays
traj = probes.rollout(agent, obs_rms, pcfg, {}, n_trials=256, seed=123)   # {} = the task distribution
{k: np.shape(v) for k, v in traj.items() if isinstance(v, np.ndarray)}

In [ ]:
probes.outcome_breakdown(traj)   # the competence gate: everything below is read only if this passes

In [ ]:
plots.plot_example_trials(traj, OUT / "example_trials.png")
from IPython.display import Image
Image(str(OUT / "example_trials.png"))

## 7 · Probe: does the reach to T1 depend on where T2 is?

In [ ]:
src(probes.preview_probe)

In [ ]:
spec, design = probes.preview_probe(pcfg)          # T1 fixed, T2 swept, preview on/off
traj = probes.rollout(agent, obs_rms, pcfg, spec, n_trials=len(design["t1"]), seed=7)
co = metrics.co_articulation_by_preview(traj, design)
{"CI (T2 visible)": co["preview_on"]["ci"], "p": co["preview_on"]["ci_p"],
 "CI (T2 hidden)": co["preview_off"]["ci"], "delta": co["delta_ci"]}

In [ ]:
plots.plot_coarticulation(traj, design, co, OUT / "coarticulation.png")
Image(str(OUT / "coarticulation.png"))

Full battery + figures + `results.json`:

```bash
python 1D/scripts/analyze.py --run 1D/runs/baseline_s1
```

## 8 · Watch it — `seqreach/video.py`

In [ ]:
gif = video.record_video(agent, obs_rms, cfg, OUT / "rollout.gif", n_trials=2)
Image(str(gif))

## 9 · Ablations are one-line overrides — `scripts/run_ablations.py`

In [ ]:
ABLATIONS = {                                        # name -> dotted overrides (full list: scripts/run_ablations.py)
    "gamma097_sparse": {"ppo.gamma": 0.97, "reward.shaping": "none"},   # discount horizon ~ one reach
    "stop_at_targets": {"task.dwell_inter": 25, "task.max_steps": 700},   # forced rest at each target
    "no_effort":       {"reward.w_effort": 0.0, "reward.w_rate": 0.0},
    "no_shaping":      {"reward.shaping": "none"},
    "greedy20":        {"reward.shaping": "greedy", "reward.shaping_coef": 20.0},   # naive pull to the target
    "mlp":             {"agent.recurrent": False},
}
c = Config().override(ABLATIONS["gamma097_sparse"]); c.exp_name = "gamma097_sparse"
c.run_name, c.ppo.gamma

```bash
python 1D/scripts/run_ablations.py --timesteps 3000000 --parallel 6 --threads 1 --wandb
```

Read next: `seqreach/env.py` (task + reward) → `seqreach/ppo.py` (training loop) →
`seqreach/probes.py` + `seqreach/metrics.py` (measurement) → `seqreach/experiments.py`
(writing your own trial list) → `doc/1D_task_and_design.md`.